In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import torch

# Force CUDA re-initialisation
if torch.cuda.is_available():
    torch.cuda.init()

print(torch.cuda.device_count())  # should now be > 0

In [ ]:
import sys
import os
import pickle
import logging
import warnings
import re
from typing import Dict, List
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

import torch

import massbalancemachine as mbm

sys.path.append(os.path.join(os.getcwd(), '../../'))
from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.config_models import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.device_count() == 0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")
    
BASE_DIR = Path(cfg.dataPath) / path_cache / f"CROSS_REGION"
# make sure BASE_DIR exists
os.makedirs(BASE_DIR, exist_ok=True)
print(f"Base directory for data: {BASE_DIR}")

# Cross-regional modelling:

### Read stakes datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (SJM+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
num_measurements

In [ ]:
region_colors = [
    "red", "blue", "green", "purple", "orange", "darkred", "cadetblue",
    "darkgreen", "darkpurple", "pink", "gray", "black"
]

# Central Asia
dfs, glacier_dict_ca = add_o2region_to_dfs(dfs, '13', cfg)
o2_color_map = {
    r: region_colors[i % len(region_colors)]
    for i, r in enumerate(
        sorted(v['O2Region'] for v in glacier_dict_ca.values()
               if v['O2Region'] is not None))
}
color_map = {
    g: o2_color_map[info['O2Region']]
    for g, info in glacier_dict_ca.items() if info['O2Region'] is not None
}
m = plot_stakes_folium(dfs['13'], color_map=color_map, tooltip_col='O2Region')
m

In [ ]:
# Alaska
dfs, glacier_dict_alaska = add_o2region_to_dfs(dfs,
                                               '01',
                                               cfg,
                                               deduplicate_glaciers=True)
o2_color_map = {
    r: region_colors[i % len(region_colors)]
    for i, r in enumerate(
        sorted(v['O2Region'] for v in glacier_dict_alaska.values()
               if v['O2Region'] is not None))
}
color_map = {
    g: o2_color_map[info['O2Region']]
    for g, info in glacier_dict_alaska.items() if info['O2Region'] is not None
}
m = plot_stakes_folium(dfs['01'], color_map=color_map, tooltip_col='O2Region')
m

### Monthly datasets:

In [ ]:
SOURCE_REGIONS = ['CH', 'NOR', 'ISL', "FR"]
TARGET_REGIONS = ['IT_AT', 'SJM', 'CENTRALASIA', 'ALA']

run_flag_by_code = {
    'ALA': False,
    'CENTRALASIA': False,
    'CH': False,
    'ISL': False,
    'NOR': False,
    'SJM': False,
    'FR': False,
    'IT_AT': False
}
# Compute monthly data once per code
monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=VOIS_CLIMATE,
    vois_topographical=STATIC_COLS,
    region_codes=SOURCE_REGIONS + TARGET_REGIONS,
    run_flag_by_code=run_flag_by_code,
)

# Map O2Region into monthly_cache for CENTRALASIA
monthly_cache['CENTRALASIA']['data_monthly']['O2Region'] = (
    monthly_cache['CENTRALASIA']['data_monthly']['GLACIER'].map({
        k:
        v['O2Region']
        for k, v in glacier_dict_ca.items()
    }))
monthly_cache['CENTRALASIA']['data_monthly_aug']['O2Region'] = (
    monthly_cache['CENTRALASIA']['data_monthly_aug']['GLACIER'].map({
        k:
        v['O2Region']
        for k, v in glacier_dict_ca.items()
    }))

# Map O2Region into monthly_cache for ALA
monthly_cache['ALA']['data_monthly']['O2Region'] = (
    monthly_cache['ALA']['data_monthly']['GLACIER'].map({
        k: v['O2Region']
        for k, v in glacier_dict_alaska.items()
    }))
monthly_cache['ALA']['data_monthly_aug']['O2Region'] = (
    monthly_cache['ALA']['data_monthly_aug']['GLACIER'].map({
        k: v['O2Region']
        for k, v in glacier_dict_alaska.items()
    }))

# IT/AT split
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'IT_AT',
    subregions={
        'IT': IT_GLACIERS,
        'AT': AT_GLACIERS
    },
    drop_parent=True,
)

# Central Asia subregions (O2Region already mapped onto dfs['13'])
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'CENTRALASIA',
    subregions={
        'CA_12': {
            'o2region_col': 'O2Region',
            'values': ['1', '2']
        },
        'CA_3': {
            'o2region_col': 'O2Region',
            'values': ['3']
        },
        'CA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
    },
)

# Alaska subregions (O2Region already mapped onto dfs['01'])
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'ALA',
    subregions={
        'ALA_2': {
            'o2region_col': 'O2Region',
            'values': ['2']
        },
        'ALA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
        'ALA_6': {
            'o2region_col': 'O2Region',
            'values': ['6']
        },
    },
)

# With subregions
TARGET_REGIONS_SUB = [
    'IT', 'AT', 'SJM', 'CA_3', 'CA_4', 'ALA_2', 'ALA_4',
    'ALA_6'
]

XREG_PAIRS = [(src,
               [r for r in SOURCE_REGIONS + TARGET_REGIONS_SUB if r != src])
              for src in SOURCE_REGIONS]

# Assemble train/test pairs from cache — no ERA5 reprocessing
res_xreg_by_source, split_info_by_source = prepare_xreg_pairs_from_cache(
    cfg=cfg,
    monthly_cache=monthly_cache,
    xreg_pairs=XREG_PAIRS,
)

In [ ]:
# --- inspect padding and FROM_DATE distribution per code ---
for code, entry in monthly_cache.items():
    print(f"\n{code}:")
    print(f"  head_pad: {entry['months_head_pad']}")
    print(f"  tail_pad: {entry['months_tail_pad']}")

    # check FROM_DATE months in the raw dfs
    for rid, df in dfs.items():
        if df is None or "SOURCE_CODE" not in df.columns:
            continue
        df_code = df[df["SOURCE_CODE"].str.upper() == code]
        if len(df_code) == 0:
            continue
        from_dt = pd.to_datetime(df_code["FROM_DATE"].astype(str),
                                 format="%Y%m%d")
        to_dt = pd.to_datetime(df_code["TO_DATE"].astype(str), format="%Y%m%d")
        print(
            f"  FROM_DATE month distribution:\n{from_dt.dt.month.value_counts().sort_index()}"
        )
        print(
            f"  TO_DATE month distribution:\n{to_dt.dt.month.value_counts().sort_index()}"
        )

### Domains shifts & feature overlap:

#### Domain shift:

In [ ]:
EXCLUDE_TARGETS = {}
res_all_xreg_by_source = {}

# The 4 individual targets we always want (each source excludes itself)
ALL_TARGETS = SOURCE_REGIONS + TARGET_REGIONS_SUB

for src_region in SOURCE_REGIONS:
    # Exclude the source region itself from targets
    target_codes = [t for t in ALL_TARGETS if t not in src_region]

    res_all_xreg = build_xreg_res_all(
        res_xreg=res_xreg_by_source[src_region],
        target_source_codes=target_codes,
        source_col="SOURCE_CODE",
        key_prefix=f"XREG_{src_region}_TO",
        ch_code=src_region,
        region_groups={},  # no group regions anymore
    )

    # Filter out excluded targets
    if EXCLUDE_TARGETS:
        res_all_xreg = {
            k: v
            for k, v in res_all_xreg.items()
            if not any(tgt in k for tgt in EXCLUDE_TARGETS)
        }

    res_all_xreg_by_source[src_region] = res_all_xreg
    print(f"{src_region} -> keys:", list(res_all_xreg.keys()))

In [ ]:
RECOMPUTE_SHIFTS = False

SHIFTS_CACHE = BASE_DIR / "domain_shifts_cache.pkl"

if RECOMPUTE_SHIFTS and SHIFTS_CACHE.exists():
    SHIFTS_CACHE.unlink()
    print(f"Deleted existing cache: {SHIFTS_CACHE}")

if SHIFTS_CACHE.exists():
    with open(SHIFTS_CACHE, "rb") as f:
        cache = pickle.load(f)
    scaler_m = cache["scaler_m"]
    scaler_s = cache["scaler_s"]
    blur_m = cache["blur_m"]
    blur_s = cache["blur_s"]
    blur_joint = cache["blur_joint"]
    bandwidths_m = cache["bandwidths_m"]
    bandwidths_s = cache["bandwidths_s"]
    all_shifts_by_source = cache["all_shifts_by_source"]
    print(
        f"Blurs from cache: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
    )
    print(f"Loaded shifts from cache: {SHIFTS_CACHE}")

else:
    scaler_m, scaler_s, scaler_joint = build_global_scalers_multi_source_simple(
        res_xreg_by_source,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
    )

    blur_m, blur_s, blur_joint = estimate_global_bandwidths_simple(
        res_xreg_by_source,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        scaler_m=scaler_m,
        scaler_s=scaler_s,
        seed=cfg.seed,
    )
    bandwidths_m = [blur_m * 0.5, blur_m, blur_m * 2.0]
    bandwidths_s = [blur_s * 0.5, blur_s, blur_s * 2.0]
    print(
        f"Estimated blurs: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
    )

    all_shifts_by_source = {}
    for src_region in SOURCE_REGIONS:
        all_shifts = {}
        for key in tqdm(res_all_xreg_by_source[src_region], desc=src_region):
            shift = compute_domain_shift(
                df_src=res_all_xreg_by_source[src_region][key]["df_train"],
                df_tgt=res_all_xreg_by_source[src_region][key]["df_test"],
                monthly_cols=MONTHLY_COLS,
                static_cols=STATIC_COLS,
                scaler_m=scaler_m,
                scaler_s=scaler_s,
                scaler_joint=scaler_joint,
                blur_m=blur_m,
                blur_s=blur_s,
                blur_joint=blur_joint,
                bandwidths_m=bandwidths_m,
                bandwidths_s=bandwidths_s,
                seed=cfg.seed,
            )
            all_shifts[key] = shift
        all_shifts_by_source[src_region] = all_shifts

    with open(SHIFTS_CACHE, "wb") as f:
        pickle.dump(
            {
                "scaler_m": scaler_m,
                "scaler_s": scaler_s,
                "blur_m": blur_m,
                "blur_s": blur_s,
                "blur_joint": blur_joint,
                "bandwidths_m": bandwidths_m,
                "bandwidths_s": bandwidths_s,
                "all_shifts_by_source": all_shifts_by_source,
            }, f)
    print(f"Saved shifts to cache: {SHIFTS_CACHE}")

# Plot summary across regions, one figure per source
for src_region, all_shifts in all_shifts_by_source.items():
    print(f"\nDomain shift summary for source: {src_region}")
    fig = plot_domain_shift_across_regions(all_shifts, src_region=src_region)
    plt.show()

In [ ]:
for key in res_all_xreg:
    Num_measurements = res_all_xreg[key]['df_test']['ID'].nunique()
    print(key, 'Target region num_measurements:', Num_measurements)

In [ ]:
selected_cols = MONTHLY_COLS + [
    c for c in STATIC_COLS if c != "ELEVATION_DIFFERENCE"
] + ["POINT_BALANCE"]

glaciers_to_plot = {
    f"FR": {
        "df": res_all_xreg_by_source['CH']['XREG_CH_TO_FR']["df_test"],
        "color": COLORS['FR'],
    },
    f"CH ": {
        "df": res_all_xreg_by_source['CH']['XREG_CH_TO_NOR']["df_train"],
        "color": COLORS['CH'],
    },
    f"ISL": {
        "df": res_all_xreg_by_source['ISL']['XREG_ISL_TO_NOR']["df_train"],
        "color": COLORS['ISL'],
    },
    f"NOR": {
        "df": res_all_xreg_by_source['NOR']['XREG_NOR_TO_ISL']["df_train"],
        "color": COLORS['NOR'],
    },
}

plot_kde_pair(
    glaciers_to_plot=glaciers_to_plot,
    selected_cols=selected_cols,
)

## ML models
### Model datasets:

In [ ]:
logs_cache_dir = BASE_DIR / "datasets"
outputs_xreg_by_source = {}

# NaN check on augmented dataframes before building LSTM datasets
print("--- NaN check on augmented dataframes ---")
for src_region, res_xreg in res_xreg_by_source.items():
    for split_name, df in [("df_train_aug", res_xreg["df_train_aug"]),
                           ("df_test_aug", res_xreg["df_test_aug"])]:
        feat_cols = [c for c in MONTHLY_COLS + STATIC_COLS if c in df.columns]
        n_nan_feat = df[feat_cols].isna().sum().sum()
        n_nan_tgt = df["POINT_BALANCE"].isna().sum(
        ) if "POINT_BALANCE" in df.columns else "N/A"
        print(
            f"  {src_region} {split_name}: feature NaNs={n_nan_feat}, target NaNs={n_nan_tgt}"
        )

for src_region in SOURCE_REGIONS:
    target_codes = [
        t for t in SOURCE_REGIONS + TARGET_REGIONS_SUB if t != src_region
    ]

    outputs_xreg_by_source[src_region] = build_or_load_lstm_all_xreg(
        cfg=cfg,
        res_xreg=res_xreg_by_source[src_region],
        MONTHLY_COLS=MONTHLY_COLS,
        STATIC_COLS=STATIC_COLS,
        cache_dir=logs_cache_dir / src_region,
        ch_code=src_region,
        target_source_codes=target_codes,
        region_groups={},  # no group regions anymore
    )
    print(f"{src_region} -> LSTM dataset keys:",
          list(outputs_xreg_by_source[src_region].keys()))

    # NaN/Inf check on the resulting datasets
    print(f"--- NaN/Inf check for {src_region} ---")
    for exp_key, assets in outputs_xreg_by_source[src_region].items():
        for ds_name in ["ds_train", "ds_test"]:
            ds = assets.get(ds_name)
            if ds is None:
                continue
            # Check directly on the underlying tensors (no iteration needed)
            checks = {"x_m": ds.Xm, "x_s": ds.Xs, "y": ds.y}
            any_issue = False
            for name, t in checks.items():
                n_nan = torch.isnan(t).sum().item()
                n_inf = torch.isinf(t).sum().item()
                if n_nan > 0 or n_inf > 0:
                    print(
                        f"  {exp_key} {ds_name} {name}: NaNs={n_nan}, Infs={n_inf}"
                    )
                    any_issue = True
            if not any_issue:
                print(f"  {exp_key} {ds_name}: OK")

### Train model:

#### LSTM:

In [ ]:
models_by_source_lstm = {}
infos_by_source_lstm = {}

TRAIN_FLAG = False  # set to True to retrain from scratch

for src_region in SOURCE_REGIONS:
    # Pick any key to get the train assets — they're the same across all targets
    any_key = next(iter(outputs_xreg_by_source[src_region]))
    train_assets = outputs_xreg_by_source[src_region][any_key]

    model, model_path, info = train_or_load_one_source_model(
        cfg=cfg,
        key=src_region,
        lstm_assets=train_assets,
        best_params=PARAMS_LSTM,
        device=device,
        models_dir=BASE_DIR / "models" / src_region,
        prefix=f"lstm_xreg",
        train_flag=TRAIN_FLAG,
        force_retrain=False,
        epochs=N_EPOCHS,
        date=MODEL_DATE,
        model_type="lstm",
    )

    models_by_source_lstm[src_region] = model
    infos_by_source_lstm[src_region] = {
        "model_path": model_path,
        **(info or {})
    }
    print(f"{src_region} -> model trained/loaded from {model_path}")

#### Transformer:

In [ ]:
models_by_source_transformer = {}
infos_by_source_transformer = {}

TRAIN_FLAG = False

for src_region in SOURCE_REGIONS:
    any_key = next(iter(outputs_xreg_by_source[src_region]))
    train_assets = outputs_xreg_by_source[src_region][any_key]

    model, model_path, info = train_or_load_one_source_model(
        cfg=cfg,
        key=src_region,
        lstm_assets=train_assets,
        best_params=PARAMS_TRANSFORMER,
        device=device,
        models_dir=BASE_DIR / "models" / src_region,
        prefix=f"transformer_xreg",
        train_flag=TRAIN_FLAG,
        force_retrain=False,
        epochs=N_EPOCHS,
        date=MODEL_DATE,
        model_type="transformer",
    )

    models_by_source_transformer[src_region] = model
    infos_by_source_transformer[src_region] = {
        "model_path": model_path,
        **(info or {})
    }
    print(f"{src_region} -> model trained/loaded from {model_path}")

## Comparison of models:

### Scatter examples:

In [ ]:
# pairs format: list of (src, tgt) tuples
pairs = [
    ("CH", "NOR"),
    ("CH", "ISL"),
    ("NOR", "FR"),
    ("NOR", "ALA_6"),
]

TARGET_LABELS = {**REGION_LABELS}

# Precompute per-pair axis limits from test data
pair_lims = {}
for src, tgt in pairs:
    key = f"XREG_{src}_TO_{tgt}"
    assets = outputs_xreg_by_source[src][key]
    vals = assets["ds_test"].y.cpu().numpy().flatten()
    vals = vals[~np.isnan(vals)]
    pad = (vals.max() - vals.min()) * 0.3
    pair_lims[(src, tgt)] = (float(vals.min() - pad), float(vals.max() + pad))

ncols = len(pairs)
fig, axes = plt.subplots(
    2,
    ncols,
    figsize=(4 * ncols, (90 * 2) / 25.4),
    sharex="col",
    sharey="col",
)
axes = np.array(axes).reshape(2, ncols)

for row_i, (model_label, models_by_source) in enumerate([
    ("LSTM", models_by_source_lstm),
    ("Transformer", models_by_source_transformer),
]):
    for col_i, (src, tgt) in enumerate(pairs):
        ax = axes[row_i, col_i]
        key = f"XREG_{src}_TO_{tgt}"
        lim = pair_lims[(src, tgt)]
        model = models_by_source[src]

        out, test_df_preds, created_fig, ax = evaluate_one_model(
            cfg=cfg,
            model=model,
            device=device,
            lstm_assets_for_key=outputs_xreg_by_source[src][key],
            ax=ax,
            ax_xlim=lim,
            ax_ylim=lim,
            legend_fontsize=NATURE_SPECS["font_min_pt"],
        )
        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=True)
        leg = ax.get_legend()
        if leg:
            leg.remove()

        row_letter = chr(ord('a') + row_i)
        panel_label = f"{row_letter}{col_i + 1}"
        ax.text(
            0.02,
            0.98,
            f"({panel_label})",
            transform=ax.transAxes,
            fontsize=NATURE_SPECS["font_max_pt"],
            #fontweight="bold",
            va="top",
            ha="left")

        if row_i == 0:
            ax.set_title(
                f"{REGION_LABELS.get(src, src)} → {REGION_LABELS.get(tgt, tgt)}",
                fontsize=NATURE_SPECS["font_max_pt"] + 1,
                #fontweight="bold",
            )
        if col_i == 0:
            ax.set_ylabel(f"{model_label}\nModeled PMB (m w.e.)",
                          fontsize=NATURE_SPECS["font_max_pt"])
        else:
            ax.set_ylabel("")
        if row_i == 0:
            ax.set_xlabel("")

fig.legend(
    handles=[
        mpatches.Patch(color=mbm.plots.COLOR_ANNUAL, label="Annual"),
        mpatches.Patch(color=mbm.plots.COLOR_WINTER, label="Winter"),
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, 0.02),
    ncols=2,
    frameon=False,
    fontsize=NATURE_SPECS["font_max_pt"],
)
# fig.suptitle(
#     "Zero-shot transfer — selected pairs  |  LSTM (top) vs Transformer (bottom)",
#     fontsize=NATURE_SPECS["font_max_pt"] + 2,
# )
fig.tight_layout(rect=[0, 0.06, 1, 1])
fig.subplots_adjust(top=0.90)
fig.savefig(
    "figures/paperTF/fig_zeroshot_selected_pairs_lstm_vs_transformer.png",
    dpi=NATURE_SPECS["dpi"],
    bbox_inches="tight",
)
plt.show()

### Scatter all:

#### CH:

In [ ]:
src = "CH"
all_targets = [
    "FR",
    "AT",
    "IT",
    "NOR",
    "ISL",
    "SJM",
    "ALA_2",
    "ALA_4",
    "ALA_6",
    "CA_3",
    "CA_4",
]
MAX_COLS = 4
n_targets = len(all_targets)
n_cols = MAX_COLS
n_target_rows = math.ceil(n_targets / n_cols)
TARGET_LABELS = {**REGION_LABELS}

# Precompute per-target axis limits
target_lims = {}
for tgt in all_targets:
    key = f"XREG_{src}_TO_{tgt}"
    vals = outputs_xreg_by_source[src][key]["ds_test"].y.cpu().numpy().flatten(
    )
    vals = vals[~np.isnan(vals)]
    pad = (vals.max() - vals.min()) * 0.3
    target_lims[tgt] = (float(vals.min() - pad), float(vals.max() + pad))

fig = plt.figure(figsize=(4 * n_cols, 3.8 * 2 * n_target_rows))
outer_gs = GridSpec(
    n_target_rows,
    1,
    figure=fig,
    hspace=0.2,  # space between blocks — reduce this
    top=0.94,  # less empty space at top
    bottom=0.04,
    left=0.08,
    right=0.98,
)

inner_gs_list = []
for block_i in range(n_target_rows):
    inner_gs = GridSpecFromSubplotSpec(
        2,
        n_cols,
        subplot_spec=outer_gs[block_i],
        hspace=0.10,  # very tight between LSTM and Transformer
        wspace=0.32,
    )
    inner_gs_list.append(inner_gs)

# Build axes array[block_i][model_i][col]
axes_grid = []
for block_i in range(n_target_rows):
    block_axes = []
    for model_i in range(2):
        row_axes = []
        for col in range(n_cols):
            ax = fig.add_subplot(inner_gs_list[block_i][model_i, col])
            row_axes.append(ax)
        block_axes.append(row_axes)
    axes_grid.append(block_axes)

for tgt_idx, tgt in enumerate(all_targets):
    block_i = tgt_idx // n_cols
    col = tgt_idx % n_cols
    lim = target_lims[tgt]
    key = f"XREG_{src}_TO_{tgt}"

    for model_i, (model_label, models_by_source) in enumerate([
        ("LSTM", models_by_source_lstm),
        ("Transformer", models_by_source_transformer),
    ]):
        ax = axes_grid[block_i][model_i][col]
        model = models_by_source[src]

        out, test_df_preds, created_fig, ax = evaluate_one_model(
            cfg=cfg,
            model=model,
            device=device,
            lstm_assets_for_key=outputs_xreg_by_source[src][key],
            ax=ax,
            ax_xlim=lim,
            ax_ylim=lim,
            legend_fontsize=NATURE_SPECS["font_min_pt"],
        )
        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=True)
        leg = ax.get_legend()
        if leg:
            leg.remove()

        # subplot label
        global_row = block_i * 2 + model_i
        row_letter = chr(ord('a') + global_row)
        panel_label = f"{row_letter}{col + 1}"
        ax.text(
            0.02,
            0.98,
            f"({panel_label})",
            transform=ax.transAxes,
            fontsize=NATURE_SPECS["font_max_pt"],
            #fontweight="bold",
            va="top",
            ha="left")

        if model_i == 0:
            ax.set_title(TARGET_LABELS.get(tgt, tgt),
                         fontsize=NATURE_SPECS["font_max_pt"] + 1,
                         fontweight="bold")
        if col == 0:
            ax.set_ylabel(f"{model_label}\nModeled PMB (m w.e.)",
                          fontsize=NATURE_SPECS["font_max_pt"])
        else:
            ax.set_ylabel("")
        if model_i == 0:
            ax.set_xlabel("")

# hide unused axes in last block
for empty_idx in range(n_targets, n_target_rows * n_cols):
    block_i = empty_idx // n_cols
    col = empty_idx % n_cols
    for model_i in range(2):
        axes_grid[block_i][model_i][col].set_visible(False)

fig.legend(
    handles=[
        mpatches.Patch(color=mbm.plots.COLOR_ANNUAL, label="Annual"),
        mpatches.Patch(color=mbm.plots.COLOR_WINTER, label="Winter"),
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, 0.0),
    ncols=2,
    frameon=False,
    fontsize=NATURE_SPECS["font_max_pt"],
)
fig.suptitle(
    f"Zero-shot transfer — source: {src}  |  LSTM (top) vs Transformer (bottom)",
    fontsize=NATURE_SPECS["font_max_pt"] + 2,
    # y removed — sits just above top=0.94
)
fig.savefig(
    f"figures/paperTF/appendix/fig_appendix_zeroshot_{src}_lstm_vs_transformer_all.png",
    dpi=NATURE_SPECS["dpi"],
    bbox_inches="tight")
plt.show()

#### NOR:

In [ ]:
src = "NOR"
all_targets = [
    "CH",
    "FR",
    "AT",
    "IT",
    "ISL",
    "SJM",
    "ALA_2",
    "ALA_4",
    "ALA_6",
    "CA_3",
    "CA_4",
]
MAX_COLS = 4
n_targets = len(all_targets)
n_cols = MAX_COLS
n_target_rows = math.ceil(n_targets / n_cols)
TARGET_LABELS = {**REGION_LABELS}

# Precompute per-target axis limits
target_lims = {}
for tgt in all_targets:
    key = f"XREG_{src}_TO_{tgt}"
    vals = outputs_xreg_by_source[src][key]["ds_test"].y.cpu().numpy().flatten(
    )
    vals = vals[~np.isnan(vals)]
    pad = (vals.max() - vals.min()) * 0.3
    target_lims[tgt] = (float(vals.min() - pad), float(vals.max() + pad))

fig = plt.figure(figsize=(4 * n_cols, 3.8 * 2 * n_target_rows))
outer_gs = GridSpec(
    n_target_rows,
    1,
    figure=fig,
    hspace=0.2,  # space between blocks — reduce this
    top=0.94,  # less empty space at top
    bottom=0.04,
    left=0.08,
    right=0.98,
)

inner_gs_list = []
for block_i in range(n_target_rows):
    inner_gs = GridSpecFromSubplotSpec(
        2,
        n_cols,
        subplot_spec=outer_gs[block_i],
        hspace=0.10,  # very tight between LSTM and Transformer
        wspace=0.32,
    )
    inner_gs_list.append(inner_gs)

# Build axes array[block_i][model_i][col]
axes_grid = []
for block_i in range(n_target_rows):
    block_axes = []
    for model_i in range(2):
        row_axes = []
        for col in range(n_cols):
            ax = fig.add_subplot(inner_gs_list[block_i][model_i, col])
            row_axes.append(ax)
        block_axes.append(row_axes)
    axes_grid.append(block_axes)

for tgt_idx, tgt in enumerate(all_targets):
    block_i = tgt_idx // n_cols
    col = tgt_idx % n_cols
    lim = target_lims[tgt]
    key = f"XREG_{src}_TO_{tgt}"

    for model_i, (model_label, models_by_source) in enumerate([
        ("LSTM", models_by_source_lstm),
        ("Transformer", models_by_source_transformer),
    ]):
        ax = axes_grid[block_i][model_i][col]
        model = models_by_source[src]

        out, test_df_preds, created_fig, ax = evaluate_one_model(
            cfg=cfg,
            model=model,
            device=device,
            lstm_assets_for_key=outputs_xreg_by_source[src][key],
            ax=ax,
            ax_xlim=lim,
            ax_ylim=lim,
            legend_fontsize=NATURE_SPECS["font_min_pt"],
        )
        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=True)
        leg = ax.get_legend()
        if leg:
            leg.remove()

        # subplot label
        global_row = block_i * 2 + model_i
        row_letter = chr(ord('a') + global_row)
        panel_label = f"{row_letter}{col + 1}"
        ax.text(
            0.02,
            0.98,
            f"({panel_label})",
            transform=ax.transAxes,
            fontsize=NATURE_SPECS["font_max_pt"],
            #fontweight="bold",
            va="top",
            ha="left")

        if model_i == 0:
            ax.set_title(TARGET_LABELS.get(tgt, tgt),
                         fontsize=NATURE_SPECS["font_max_pt"] + 1,
                         fontweight="bold")
        if col == 0:
            ax.set_ylabel(f"{model_label}\nModeled PMB (m w.e.)",
                          fontsize=NATURE_SPECS["font_max_pt"])
        else:
            ax.set_ylabel("")
        if model_i == 0:
            ax.set_xlabel("")

# hide unused axes in last block
for empty_idx in range(n_targets, n_target_rows * n_cols):
    block_i = empty_idx // n_cols
    col = empty_idx % n_cols
    for model_i in range(2):
        axes_grid[block_i][model_i][col].set_visible(False)

fig.legend(
    handles=[
        mpatches.Patch(color=mbm.plots.COLOR_ANNUAL, label="Annual"),
        mpatches.Patch(color=mbm.plots.COLOR_WINTER, label="Winter"),
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, 0.0),
    ncols=2,
    frameon=False,
    fontsize=NATURE_SPECS["font_max_pt"],
)
fig.suptitle(
    f"Zero-shot transfer — source: {src}  |  LSTM (top) vs Transformer (bottom)",
    fontsize=NATURE_SPECS["font_max_pt"] + 2,
    # y removed — sits just above top=0.94
)
fig.savefig(
    f"figures/paperTF/appendix/fig_appendix_zeroshot_{src}_lstm_vs_transformer_all.png",
    dpi=NATURE_SPECS["dpi"],
    bbox_inches="tight")
plt.show()

### Barplot:

In [ ]:
df_metrics_by_source_lstm = {}

# Preferred display order for target regions — any not listed appear at the end
DISPLAY_ORDER = [
    'CH',
    'FR',
    'IT',
    'NOR',
    'ISL',
    'SJM',
    'ALA',
    'CENTRALASIA',
]

EXCLUDE_TARGETS = {'CENTRALASIA', 'ALA'}

for src_region in SOURCE_REGIONS:
    print(f"\nEvaluating source region: {src_region}")
    target_keys = [
        k for k in outputs_xreg_by_source[src_region].keys()
        if k.split("_TO_")[-1] not in EXCLUDE_TARGETS
    ]
    models_by_key = {
        key: models_by_source_lstm[src_region]
        for key in target_keys
    }

    available_targets = [k.split("_TO_")[-1] for k in target_keys]
    custom_order = [t for t in DISPLAY_ORDER if t in available_targets]
    custom_order += [t for t in available_targets if t not in custom_order]

    df_metrics = evaluate_all_models_metrics_only(
        cfg=cfg,
        models_by_key=models_by_key,
        lstm_assets_by_key=outputs_xreg_by_source[src_region],
        device=device,
        complement_key=f'XREG_{src_region}_TO_',
        custom_order=custom_order,
    )

    df_metrics_by_source_lstm[src_region] = df_metrics

In [ ]:
df_metrics_by_source_transformer = {}

for src_region in SOURCE_REGIONS:
    print(f"\nEvaluating source region: {src_region}")
    target_keys = [
        k for k in outputs_xreg_by_source[src_region].keys()
        if k.split("_TO_")[-1] not in EXCLUDE_TARGETS
    ]
    models_by_key = {
        key: models_by_source_transformer[src_region]
        for key in target_keys
    }

    available_targets = [k.split("_TO_")[-1] for k in target_keys]
    custom_order = [t for t in DISPLAY_ORDER if t in available_targets]
    custom_order += [t for t in available_targets if t not in custom_order]

    df_metrics_by_source_transformer[
        src_region] = evaluate_all_models_metrics_only(
            cfg=cfg,
            models_by_key=models_by_key,
            lstm_assets_by_key=outputs_xreg_by_source[src_region],
            device=device,
            complement_key=f'XREG_{src_region}_TO_',
            custom_order=custom_order,
        )

In [ ]:
TF_COLOR = NATURE_PALETTE["sky_blue"]
LSTM_COLOR = "#aaaaaa"

METRIC_LABELS = {
    "RMSE_annual": "RMSE annual\n(m w.e.)",
    "RMSE_winter": "RMSE winter\n(m w.e.)",
    "RMSE_summer": "RMSE summer\n(m w.e.)",
    "MAE_annual": "MAE annual\n(m w.e.)",
    "R2_annual": "R² annual",
    "R2_winter": "R² winter",
    "pearson_annual": "Pearson r annual",
}

SOURCE_ORDER = ["CH", "FR", "NOR", "ISL", "ALA", "SJM"]
TARGET_ORDER = [
    "CH",
    "FR",
    "IT",
    "AT",
    "NOR",
    "ISL",
    "ALA",
    "SJM",
]


def _parse_key(key):
    m = re.match(r"XREG_(.+)_TO_(.+)", key)
    return (m.group(1), m.group(2)) if m else (key, key)


def _sort_key(val, order):
    try:
        return order.index(val)
    except ValueError:
        return len(order)


# ── inputs ────────────────────────────────────────────────────────────────────
metrics = ["RMSE_annual", "RMSE_winter"]
exclude_targets = []

# ── build long DataFrame ──────────────────────────────────────────────────────
rows = []
for model_label, metrics_dict in [("LSTM", df_metrics_by_source_lstm),
                                  ("Transformer",
                                   df_metrics_by_source_transformer)]:
    for src, df in metrics_dict.items():
        for key, row in df.iterrows():
            _, tgt = _parse_key(key)
            if tgt in exclude_targets:
                continue
            entry = {"source": src, "target": tgt, "model": model_label}
            entry.update({m: row[m] for m in metrics if m in row.index})
            rows.append(entry)

df_all = pd.DataFrame(rows)

# ── compute delta: LSTM − Transformer ────────────────────────────────────────
# positive = LSTM is worse (for RMSE), i.e. Transformer wins
# negative = Transformer is worse
df_lstm = df_all[df_all["model"] == "LSTM"].set_index(["source", "target"])
df_tf = df_all[df_all["model"] == "Transformer"].set_index(
    ["source", "target"])
df_delta = (df_lstm[metrics] - df_tf[metrics]).reset_index()
df_delta.columns = ["source", "target"] + metrics

sources = sorted(df_delta["source"].unique(),
                 key=lambda s: _sort_key(s, SOURCE_ORDER))
targets_per_source = {
    src:
    sorted(df_delta[df_delta["source"] == src]["target"].unique(),
           key=lambda t: _sort_key(t, TARGET_ORDER))
    for src in sources
}

In [ ]:
# ── figure ────────────────────────────────────────────────────────────────────
n_metrics, n_sources = len(metrics), len(sources)
mm2in = 1 / 25.4
fig = plt.figure(figsize=(200 * mm2in, n_metrics * 1.55 + 0.9))

# outer GS: 1 row × n_sources columns (one column per source)
outer_gs = GridSpec(1,
                    n_sources,
                    figure=fig,
                    hspace=0.0,
                    wspace=0.38,
                    left=0.11,
                    right=0.98,
                    top=0.88,
                    bottom=0.28)

# inner GS per source: n_metrics rows sharing the same x axis
inner_gs_list = [
    GridSpecFromSubplotSpec(n_metrics,
                            1,
                            subplot_spec=outer_gs[0, col_i],
                            hspace=0.08) for col_i in range(n_sources)
]

# pre-compute symmetric ylim per metric (across all sources)
ylims = {}
for metric in metrics:
    vals = df_delta[metric].dropna()
    abs_max = max(abs(vals.min()), abs(vals.max()), 0.05)
    ypad = abs_max * 0.18
    ylims[metric] = (-(abs_max + ypad), abs_max + ypad)

for col_i, src in enumerate(sources):
    tgts = targets_per_source[src]
    x = np.arange(len(tgts))
    tick_labels = [REGION_LABELS.get(t, t) for t in tgts]

    for row_i, metric in enumerate(metrics):
        is_r2 = any(k in metric for k in ("R2", "r2", "pearson"))
        is_bottom = (row_i == n_metrics - 1)

        ax = fig.add_subplot(inner_gs_list[col_i][row_i, 0])

        panel_idx = row_i * n_sources + col_i
        ax.text(
            0.03,
            0.97,
            f"({chr(ord('a') + panel_idx)})",
            transform=ax.transAxes,
            fontsize=9,
            #fontweight="bold",
            va="top",
            ha="left",
            zorder=5)

        # alternating column backgrounds
        ax.set_facecolor("#f5f5f5")
        for xi in range(0, len(tgts), 2):
            ax.axvspan(xi - 0.5, xi + 0.5, color="white", alpha=1.0, zorder=0)

        sub = df_delta[df_delta["source"] == src]
        bar_vals = [
            float(sub.loc[sub["target"] == t, metric].iloc[0])
            if not sub[sub["target"] == t].empty else np.nan for t in tgts
        ]

        tf_better = [(v > 0) if not is_r2 else (v < 0) for v in bar_vals]
        colors = [TF_COLOR if better else LSTM_COLOR for better in tf_better]

        ax.bar(x, bar_vals, width=0.6, color=colors, zorder=3, linewidth=0)
        ax.axhline(0, color="black", linewidth=0.6, zorder=4)

        ax.set_xlim(-0.5, len(tgts) - 0.5)
        ax.set_ylim(ylims[metric])
        ax.set_xticks(x)

        if is_bottom:
            ax.set_xticklabels(tick_labels,
                               fontsize=6,
                               rotation=90,
                               ha="center")
        else:
            ax.set_xticklabels([])

        ax.tick_params(axis="y", labelsize=6, width=0.5, length=2)
        ax.tick_params(axis="x", length=0)
        # ax.spines[["top", "right"]].set_visible(False)
        # ax.spines[["left", "bottom"]].set_linewidth(0.5)

        if row_i == 0:
            ax.set_title(f"Source: {REGION_LABELS.get(src, src)}",
                         fontsize=7,
                         pad=4)
        if col_i == 0:
            ax.set_ylabel(
                f"Δ {METRIC_LABELS.get(metric, metric)}\n(LSTM − TF)",
                fontsize=6.5,
                labelpad=4)

fig.legend(
    handles=[
        mpatches.Patch(color=TF_COLOR, alpha=0.88, label="Transformer better"),
        mpatches.Patch(color=LSTM_COLOR, alpha=0.88, label="LSTM better"),
    ],
    loc="lower center",
    ncol=2,
    fontsize=7,
    frameon=False,
    bbox_to_anchor=(0.54, 0.02),
    handlelength=1.2,
    handletextpad=0.4,
)
# fig.suptitle(
#     "Zero-shot cross-region — LSTM vs Transformer (Δ = LSTM − TF, positive = TF better)",
#     fontsize=8.5,
#     y=1.003,
# )

save_path = "figures/paperTF/fig_zeroshot_comparison.png"
fig.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ── Supplementary: absolute RMSE dots per source × target ────────────────────
mm2in = 1 / 25.4

fig = plt.figure(figsize=(200 * mm2in, n_metrics * 1.55 + 0.9))

outer_gs = GridSpec(1, n_sources, figure=fig,
                    hspace=0.0, wspace=0.38,
                    left=0.11, right=0.98, top=0.88, bottom=0.28)

inner_gs_list = [
    GridSpecFromSubplotSpec(n_metrics, 1,
                            subplot_spec=outer_gs[0, col_i],
                            hspace=0.08)
    for col_i in range(n_sources)
]

# pre-compute ylim per metric (across all sources, both models)
ylims_abs = {}
for metric in metrics:
    vals = df_all[metric].dropna()
    ypad = (vals.max() - vals.min()) * 0.12
    ylims_abs[metric] = (max(0, vals.min() - ypad), vals.max() + ypad)

for col_i, src in enumerate(sources):
    tgts        = targets_per_source[src]
    x           = np.arange(len(tgts))
    tick_labels = [REGION_LABELS.get(t, t) for t in tgts]

    for row_i, metric in enumerate(metrics):
        is_bottom = (row_i == n_metrics - 1)

        ax = fig.add_subplot(inner_gs_list[col_i][row_i, 0])

        panel_idx = row_i * n_sources + col_i
        ax.text(0.03, 0.97, f"({chr(ord('a') + panel_idx)})",
                transform=ax.transAxes,
                fontsize=9,
                va="top", ha="left",
                zorder=5)

        # alternating column backgrounds
        ax.set_facecolor("#f5f5f5")
        for xi in range(0, len(tgts), 2):
            ax.axvspan(xi - 0.5, xi + 0.5, color="white", alpha=1.0, zorder=0)

        for model_label, color, offset in [
            ("LSTM",        LSTM_COLOR, -0.15),
            ("Transformer", TF_COLOR,    0.15),
        ]:
            sub = df_all[(df_all["source"] == src) & (df_all["model"] == model_label)]
            dot_vals = [
                float(sub.loc[sub["target"] == t, metric].iloc[0])
                if not sub[sub["target"] == t].empty else np.nan
                for t in tgts
            ]
            ax.scatter(x + offset, dot_vals,
                       color=color, s=30, zorder=4,
                       alpha=0.88, linewidths=0)


        ax.set_xlim(-0.5, len(tgts) - 0.5)
        ax.set_ylim(ylims_abs[metric])
        ax.set_xticks(x)

        if is_bottom:
            ax.set_xticklabels(tick_labels, fontsize=6, rotation=90, ha="center")
        else:
            ax.set_xticklabels([])

        ax.tick_params(axis="y", labelsize=6, width=0.5, length=2)
        ax.tick_params(axis="x", length=0)

        if row_i == 0:
            ax.set_title(f"Source: {REGION_LABELS.get(src, src)}",
                         fontsize=7, pad=4)
        if col_i == 0:
            ax.set_ylabel(METRIC_LABELS.get(metric, metric),
                          fontsize=6.5, labelpad=4)

fig.legend(
    handles=[
        mpatches.Patch(color=LSTM_COLOR, alpha=0.88, label="LSTM"),
        mpatches.Patch(color=TF_COLOR,   alpha=0.88, label="Transformer"),
    ],
    loc="lower center",
    ncol=2,
    fontsize=7,
    frameon=False,
    bbox_to_anchor=(0.54, 0.02),
    handlelength=1.2,
    handletextpad=0.4,
)

save_path = "figures/paperTF/appendix/fig_appendix_zeroshot_comparison_absolute.png"
fig.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

## Combined figure:

In [ ]:
def to_roman(n):
    vals = [(1000,'m'),(900,'cm'),(500,'d'),(400,'cd'),(100,'c'),(90,'xc'),
            (50,'l'),(40,'xl'),(10,'x'),(9,'ix'),(5,'v'),(4,'iv'),(1,'i')]
    result = ''
    for v, s in vals:
        while n >= v:
            result += s
            n -= v
    return result

from matplotlib.gridspec import GridSpec

mm2in = 1 / 25.4

pairs = [
    ("CH", "ISL"),
    ("FR", "NOR"),
    ("NOR", "ALA_6"),
    ("ISL", "IT"),
]

N_PAIRS = len(pairs)
N_SOURCES = len(sources)
N_METRICS = len(metrics)
N_COLS = max(N_PAIRS, N_SOURCES)

# height ratios
SCATTER_ROW_H = 2.8  # inches per scatter row
SPACER_H = 1.5  # gap between scatter and barplots
BAR_ROW_H = 2.5  # inches per bar metric row

height_ratios = ([SCATTER_ROW_H] * 2 + [SPACER_H] + [BAR_ROW_H] * N_METRICS)
fig_h = sum(height_ratios) + 1.0
fig_w = max(200 * mm2in, 4.5 * N_COLS)

suplot_title_fontsize = NATURE_SPECS["font_max_pt"]+4

fig = plt.figure(figsize=(fig_w, fig_h))

gs = GridSpec(
    2 + 1 + N_METRICS,
    N_COLS,
    figure=fig,
    height_ratios=height_ratios,
    hspace=0.2,
    wspace=0.35,
    left=0.08,
    right=0.98,
    top=0.97,
    bottom=0.06,
)

# ── scatter section ───────────────────────────────────────────────────────────
pair_lims = {}
for src, tgt in pairs:
    key = f"XREG_{src}_TO_{tgt}"
    vals = outputs_xreg_by_source[src][key]["ds_test"].y.cpu().numpy().flatten(
    )
    vals = vals[~np.isnan(vals)]
    pad = (vals.max() - vals.min()) * 0.5
    pair_lims[(src, tgt)] = (float(vals.min() - pad), float(vals.max() + pad))

for row_i, (model_label, models_by_source) in enumerate([
    ("LSTM", models_by_source_lstm),
    ("Transformer", models_by_source_transformer),
]):
    for col_i, (src, tgt) in enumerate(pairs):
        ax = fig.add_subplot(gs[row_i, col_i])
        key = f"XREG_{src}_TO_{tgt}"
        lim = pair_lims[(src, tgt)]

        evaluate_one_model(
            cfg=cfg,
            model=models_by_source[src],
            device=device,
            lstm_assets_for_key=outputs_xreg_by_source[src][key],
            ax=ax,
            ax_xlim=lim,
            ax_ylim=lim,
            legend_fontsize=NATURE_SPECS["font_min_pt"],
        )
        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=True)
        leg = ax.get_legend()
        if leg:
            leg.remove()

        # panel_idx = row_i * N_PAIRS + col_i
        # ax.text(0.02,
        #         0.98,
        #         f"({chr(ord('a') + panel_idx)})",
        panel_idx = row_i * N_PAIRS + col_i + 1  # 1-indexed for roman
        ax.text(0.02, 0.98, f"({to_roman(panel_idx)})",
                transform=ax.transAxes,
                fontsize=NATURE_SPECS["font_max_pt"],
                va="top",
                ha="left")

        if row_i == 0:
            ax.set_title(
                f"{REGION_LABELS.get(src, src)} → {REGION_LABELS.get(tgt, tgt)}",
                fontsize=suplot_title_fontsize,
            )
        if col_i == 0:
            ax.set_ylabel(f"{model_label}\nModeled PMB (m w.e.)",
                          fontsize=NATURE_SPECS["font_max_pt"])
        else:
            ax.set_ylabel("")
        # suppress x label — no room between sections
        ax.set_xlabel("")

# row 2 = spacer, no axes

# ── barplot section ───────────────────────────────────────────────────────────
ylims = {}
for metric in metrics:
    vals = df_delta[metric].dropna()
    abs_max = max(abs(vals.min()), abs(vals.max()), 0.05)
    ypad = abs_max * 0.18
    ylims[metric] = (-(abs_max + ypad), abs_max + ypad)

bar_row_offset = 3
bar_letter_offset = 2 * N_PAIRS

for col_i, src in enumerate(sources):
    tgts = targets_per_source[src]
    x = np.arange(len(tgts))
    tick_labels = [REGION_LABELS.get(t, t) for t in tgts]

    for row_i, metric in enumerate(metrics):
        is_r2 = any(k in metric for k in ("R2", "r2", "pearson"))
        is_bottom = (row_i == N_METRICS - 1)

        ax = fig.add_subplot(gs[bar_row_offset + row_i, col_i])

        # panel_idx = 2 * N_PAIRS + row_i * N_SOURCES + col_i
        # ax.text(0.03,
        #         0.97,
        #         f"({chr(ord('a') + panel_idx)})",
        panel_idx = 2 * N_PAIRS + row_i * N_SOURCES + col_i + 1
        ax.text(0.03, 0.97, f"({to_roman(panel_idx)})",
                transform=ax.transAxes,
                fontsize=NATURE_SPECS["font_max_pt"],
                va="top",
                ha="left",
                zorder=5)

        ax.set_facecolor("#f5f5f5")
        for xi in range(0, len(tgts), 2):
            ax.axvspan(xi - 0.5, xi + 0.5, color="white", alpha=1.0, zorder=0)

        sub = df_delta[df_delta["source"] == src]
        bar_vals = [
            float(sub.loc[sub["target"] == t, metric].iloc[0])
            if not sub[sub["target"] == t].empty else np.nan for t in tgts
        ]
        tf_better = [(v > 0) if not is_r2 else (v < 0) for v in bar_vals]
        bar_colors = [TF_COLOR if b else LSTM_COLOR for b in tf_better]

        ax.bar(x, bar_vals, width=0.6, color=bar_colors, zorder=3, linewidth=0)
        ax.axhline(0, color="black", linewidth=0.6, zorder=4)
        ax.set_xlim(-0.5, len(tgts) - 0.5)
        ax.set_ylim(ylims[metric])
        ax.set_xticks(x)

        if is_bottom:
            ax.set_xticklabels(tick_labels,
                               fontsize=NATURE_SPECS["font_min_pt"],
                               rotation=90,
                               ha="center")
        else:
            ax.set_xticklabels([])

        ax.tick_params(axis="y",
                       labelsize=NATURE_SPECS["font_min_pt"],
                       width=0.5,
                       length=2)
        ax.tick_params(axis="x", length=0)

        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=True)
        if row_i == 0:
            ax.set_title(f"Source: {REGION_LABELS.get(src, src)}",
                         fontsize=suplot_title_fontsize,
                         pad=4)
        if col_i == 0:
            ax.set_ylabel(
                f"Δ {METRIC_LABELS.get(metric, metric)}\n(LSTM − TF)",
                fontsize=NATURE_SPECS["font_max_pt"],
                labelpad=4)
        ax.set_facecolor("#f5f5f5")  # reapply after nature style reset

fig.savefig(
    "figures/paperTF/fig_zeroshot_combined.png",
    dpi=NATURE_SPECS["dpi"],
    bbox_inches="tight",
)
plt.show()

# ── save legend as separate figure ───────────────────────────────────────────
fig_legend, ax_legend = plt.subplots(figsize=(6, 0.5))
ax_legend.axis("off")
fig_legend.legend(
    handles=[
        mpatches.Patch(color=mbm.plots.COLOR_ANNUAL, label="Annual PMB"),
        mpatches.Patch(color=mbm.plots.COLOR_WINTER, label="Winter PMB"),
        mpatches.Patch(color=TF_COLOR,
                       alpha=0.88,
                       label="Transformer better (Δ)"),
        mpatches.Patch(color=LSTM_COLOR, alpha=0.88, label="LSTM better (Δ)"),
    ],
    loc="center",
    ncol=4,
    fontsize=NATURE_SPECS["font_max_pt"],
    frameon=False,
    handlelength=1.2,
    handletextpad=0.4,
)
fig_legend.savefig(
    "figures/paperTF/fig_zeroshot_combined_legend.png",
    dpi=NATURE_SPECS["dpi"],
    bbox_inches="tight",
)
plt.close(fig_legend)